# Bài học 15: Agent Nghiên Cứu và Dàn Ý

Trong bài học này, chúng ta sẽ xây dựng **2 agent đầu tiên** của pipeline SEO:

1. **Research Agent** — Tìm kiếm trên web, phân tích từ khóa và nội dung đối thủ
2. **Outline Agent** — Tạo dàn ý có cấu trúc từ kết quả nghiên cứu

Đây là những agent thực tế từ sản phẩm của chúng ta. Luồng xử lý:

```
Chủ đề --> [Research Agent] --> Ghi chú nghiên cứu --> [Outline Agent] --> ContentOutline (JSON)
```

Research Agent sử dụng **DuckDuckGoTools** để tìm kiếm web. Outline Agent sử dụng **output_schema** để trả về JSON có cấu trúc thay vì văn bản tự do.

> **Lưu ý**: Claude (Anthropic) hỗ trợ sử dụng `tools` và `output_schema` cùng lúc. Như bạn đã học ở Bài học 7, đây là một trong những lý do chính ta chọn Claude Sonnet cho các agent này.

## Từ Mini Pipeline đến Sản Phẩm Thực Tế

Ở Bài học 13, bạn đã xây dựng một mini pipeline với schema lồng nhau (`Section` bên trong `DetailedOutline`). Pipeline sản phẩm sử dụng **cùng một pattern** — chỉ với nhiều trường hơn và hướng dẫn chi tiết hơn.

Schema `ContentOutline` bên dưới sử dụng `OutlineSection` lồng bên trong, giống hệt `Section` bên trong `DetailedOutline` mà bạn đã thực hành. Nếu bạn đã hiểu Bài học 13, phần này sẽ rất quen thuộc.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from agno.agent import Agent
from agno.models.anthropic import Claude
from agno.tools.duckduckgo import DuckDuckGoTools
from pydantic import BaseModel, Field

## Schema: ContentOutline

Trước khi xây dựng agent, ta cần định nghĩa **output schema** — cấu trúc dữ liệu mà Outline Agent sẽ trả về.

Đây chính là **schema dùng trong sản phẩm** (`agents/schemas.py`). Khi bạn truyền `output_schema=ContentOutline` cho agent, LLM sẽ trả về JSON khớp chính xác cấu trúc này.

Schema gồm 2 class:
- `OutlineSection` — Một phần trong bài viết (tương ứng với H2)
- `ContentOutline` — Toàn bộ dàn ý: tiêu đề, meta description, từ khóa, các phần

In [ ]:
class OutlineSection(BaseModel):
    heading: str = Field(description="Section heading (H2)")
    subheadings: list[str] = Field(default_factory=list, description="Sub-section headings (H3)")
    key_points: list[str] = Field(description="Bullet points to cover")
    seo_keywords: list[str] = Field(default_factory=list, description="Target keywords for this section")


class ContentOutline(BaseModel):
    title: str = Field(description="SEO-optimized article title")
    meta_description: str = Field(description="Meta description, max 160 chars")
    target_keywords: list[str] = Field(description="Primary SEO keywords")
    sections: list[OutlineSection] = Field(description="Ordered list of content sections")
    tone: str = Field(default="informative", description="Writing tone")

## Research Agent

Agent đầu tiên trong pipeline. Nhiệm vụ của nó:

- **Tìm kiếm trên web** bằng DuckDuckGo để thu thập thông tin về chủ đề
- **Phân tích từ khóa** — tìm từ khóa chính và phụ
- **Kiểm tra đối thủ** — nội dung nào đang xếp hạng cao, có những khoảng trống nào
- **Trả về ghi chú nghiên cứu** dưới dạng văn bản thuần (chưa cần JSON)

Agent này sử dụng `DuckDuckGoTools()` — một toolkit có sẵn của Agno cho phép agent tự động tìm kiếm web khi cần thông tin.

In [ ]:
research_agent = Agent(
    name="Research Agent",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    tools=[DuckDuckGoTools()],
    instructions=[
        "You are an expert SEO researcher.",
        "Research the given topic using web search.",
        "Identify primary and secondary keywords, analyze what top-ranking content covers, "
        "and find content gaps.",
        "Return your findings as clear, organized research notes.",
    ],
)

## Outline Agent

Agent thứ hai. Nhận ghi chú nghiên cứu từ Research Agent và tạo dàn ý có cấu trúc.

Các điểm quan trọng:
- Sử dụng `output_schema=ContentOutline` — agent **bắt buộc** trả về JSON đúng format
- **Không cần tools** — Outline Agent chỉ xử lý văn bản, không tìm kiếm web hay gọi API
- Dàn ý gồm 5-8 phần với H2, H3, các điểm chính và từ khóa cho mỗi phần

> **Nhắc lại**: `output_schema` là cách Agno buộc LLM trả về dữ liệu có cấu trúc. Thay vì văn bản tự do, bạn nhận được một object Pydantic với `.title`, `.sections`, v.v.

In [ ]:
outline_agent = Agent(
    name="Outline Agent",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    output_schema=ContentOutline,
    instructions=[
        "You are an expert content strategist.",
        "Given research notes about a topic, create a structured content outline.",
        "Include 5-8 sections with clear H2 headings, optional H3 subheadings, "
        "key points, and relevant SEO keywords per section.",
        "The title should be SEO-optimized and compelling.",
        "The meta description must be under 160 characters.",
    ],
)

## Chạy Thử: Research --> Outline

Bây giờ ta sẽ chạy cả hai agent theo thứ tự:

1. Research Agent tìm kiếm trên web về chủ đề
2. Outline Agent nhận ghi chú nghiên cứu và tạo dàn ý

Đây là **cùng một pipeline pattern** mà bạn đã xây dựng trong mini pipeline ở Bài học 13 — đầu ra của agent này là đầu vào cho agent tiếp theo. Sản phẩm thực tế thêm theo dõi database và xử lý lỗi, nhưng logic cốt lõi hoàn toàn giống nhau.

> **Chi phí:** ~$0.10-0.20 (2 lần gọi Sonnet). Mất khoảng 30-60 giây.

In [ ]:
topic = "How to optimize on-page SEO for your website"

print("Bước 1: Đang nghiên cứu...")
research = research_agent.run(f"Research this topic for an SEO article: {topic}")
print(f"Nghiên cứu xong! ({len(research.content)} ký tự)\n")

print("Bước 2: Đang tạo dàn ý...")
outline_response = outline_agent.run(
    f"Create a structured content outline from these research notes:\n\n{research.content}"
)
outline = outline_response.content

print(f"Tiêu đề: {outline.title}")
print(f"Meta: {outline.meta_description}")
print(f"Từ khóa: {', '.join(outline.target_keywords)}")
print(f"\nCác phần:")
for i, section in enumerate(outline.sections, 1):
    print(f"  {i}. {section.heading}")
    for sp in section.subheadings:
        print(f"     - {sp}")

## Bài Tập

Đổi biến `topic` thành một chủ đề liên quan đến công việc của bạn, rồi chạy lại ô chạy thử phía trên.

Sau khi dàn ý được tạo, hãy điền vào chỗ trống bên dưới để:
1. In ra tổng số phần (sections)
2. In ra các điểm chính (key points) của **phần đầu tiên**
3. In ra meta description và kiểm tra: nó có dưới 160 ký tự không? (dùng `len()`)

Đây là cách bạn kiểm tra và xác thực đầu ra của agent trong quy trình thực tế.

In [ ]:
# Bài tập: Kiểm tra dàn ý từ phần chạy thử ở trên

# 1. In ra tổng số phần (sections)
print(f"Tổng số phần: {___(outline.sections)}")

# 2. In ra các điểm chính (key points) của phần đầu tiên
first_section = outline.sections[___]
print(f"\nPhần đầu tiên: {first_section.heading}")
print(f"Các điểm chính: {first_section.___}")

# 3. In ra meta description và kiểm tra có dưới 160 ký tự không
meta = outline.___
print(f"\nMeta description: {meta}")
print(f"Độ dài: {___(meta)} ký tự")
print(f"Dưới 160 ký tự: {len(meta) ___ 160}")

<details>
<summary>Bấm để xem đáp án</summary>

```python
# Bài tập: Kiểm tra dàn ý từ phần chạy thử ở trên

# 1. In ra tổng số phần (sections)
print(f"Tổng số phần: {len(outline.sections)}")

# 2. In ra các điểm chính (key points) của phần đầu tiên
first_section = outline.sections[0]
print(f"\nPhần đầu tiên: {first_section.heading}")
print(f"Các điểm chính: {first_section.key_points}")

# 3. In ra meta description và kiểm tra có dưới 160 ký tự không
meta = outline.meta_description
print(f"\nMeta description: {meta}")
print(f"Độ dài: {len(meta)} ký tự")
print(f"Dưới 160 ký tự: {len(meta) < 160}")
```
</details>